# Download the data — **do this at home, at least two days before the session**

Everything this course uses is downloaded here, once, from its original public source. Nothing is downloaded during the session, because thirty laptops pulling data over seminar Wi-Fi is a guaranteed way to lose the first half hour.

**The whole thing is about 18 MB and takes a couple of minutes.** You do not need a fast connection.

### What you actually get

| Module | Real data downloaded | What is simulated |
|---|---|---|
| **A** MRI | OASIS-2: 373 real MRI sessions from 150 people | the 2D slice pictures |
| **C** biomarkers | *(nothing — no shareable dataset exists)* | the whole cohort |
| **D** confounding | OASIS-1: 416 real people, one scan each | nothing |
| **E** clinical records | *(nothing — EHR data is never open)* | the whole cohort |
| **F** genetics | GWAS Catalog: real AD risk variants and effect sizes | the genotypes |
| **G** transcriptomics | GEO GSE1297: 31 real post-mortem brains | nothing |
| **H** chemistry | MoleculeNet BACE-1: 1513 real compounds | nothing |

Each module's notebook prints its own provenance in its first cell, and says plainly which parts are real measurements and which are not.

### How this notebook works

1. Choose your modules · 2. See the sizes · 3. Download · 4. Prepare · 5. Verify

It only needs a plain Python install — no pandas, no scikit-learn — so you can run it before setting up the environment. Re-running is always safe: files already downloaded and verified are skipped.


In [ ]:
# Standard library only, so this works on a bare Python install.
import sys
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').exists())
sys.path.insert(0, str(repo_root / 'src'))

import download
from data_registry import MODULES, SOURCES, ACCESS_DATE, module_ids, sources_for, download_bytes

print('Ready. Modules available:', ', '.join(module_ids()))
print('Source links last verified:', ACCESS_DATE)
print('Files will be saved under:', repo_root / 'data')


## 1 · Choose your modules

Two modules is normal and enough — the one you plan to work through fully, and one to sample. Downloading everything is fine too; it is only 18 MB.

**Edit the list in the cell below.** This plain cell always works. The checkbox widget after it does the same thing, if `ipywidgets` happens to be installed — but you can ignore it entirely.


In [ ]:
# Put the modules you want here, or use 'ALL'.
MODULES_TO_FETCH = 'ALL'         # e.g. ['C', 'H']  or  ['A', 'D', 'G']  or  'ALL'

WIDGET_STATE = {}                # only used if you run the optional widget cell below


In [ ]:
# OPTIONAL: checkboxes. Skip this cell if you prefer the list above — both do the same thing.
try:
    import ipywidgets as widgets
    from IPython.display import display

    preselected = module_ids() if MODULES_TO_FETCH == 'ALL' else list(MODULES_TO_FETCH)
    boxes = {m: widgets.Checkbox(value=(m in preselected),
                                 description=f"{m} — {MODULES[m]['title']}",
                                 layout=widgets.Layout(width='520px'),
                                 style={'description_width': 'initial'})
             for m in module_ids()}
    select_all = widgets.Checkbox(value=False, description='select all')

    def _toggle_all(change):
        for box in boxes.values():
            box.value = change['new']

    def _sync(_=None):
        WIDGET_STATE.clear()
        WIDGET_STATE.update({m: box.value for m, box in boxes.items()})

    select_all.observe(_toggle_all, names='value')
    for box in boxes.values():
        box.observe(_sync, names='value')
    _sync()
    display(widgets.VBox([select_all] + list(boxes.values())))
    print('Tick what you want, then run section 3.')
except ImportError:
    print('ipywidgets is not installed — that is completely fine.')
    print('Just use the MODULES_TO_FETCH list in the cell above.')


## 2 · What you are about to download

Sizes are measured from the actual files, not estimated, so they cannot drift out of date. Read this table before you choose.


In [ ]:
print(f"{'Module':<8}{'Download':>10}   Source")
print('-' * 96)
for module in module_ids():
    entry = MODULES[module]
    keys = entry['sources']
    size = download.human(download_bytes([module])) if keys else 'none'
    first = SOURCES[keys[0]]['title'] if keys else 'generated locally (no shareable source exists)'
    print(f"{module:<8}{size:>10}   {first}")
    for key in keys[1:]:
        print(f"{'':<18}   {SOURCES[key]['title']}")
    print(f"{'':<18}   feasibility: {entry['feasibility']}")
print('-' * 96)
print(f"Everything: {download.human(download_bytes(module_ids()))} to download, "
      f"roughly 1-3 minutes on a home connection.")
print('After preparation the derived tables add about another 2 MB.')


### Licences and citations

Run this cell and read it. These are other people's data, donated by real patients and released under terms that ask for attribution in return.


In [ ]:
for key, source in SOURCES.items():
    print(f"{source['title']}")
    print(f"  what     : {source['what']}")
    print(f"  licence  : {source['licence']}")
    print(f"  cite     : {source['citation']}")
    print(f"  homepage : {source['homepage']}")
    print()


## 3 · Download

Re-running this is safe. A file that is already present and passes its checksum is skipped in a fraction of a second. Interrupted downloads are simply restarted; a partial file is never left behind. Transient network failures are retried three times.


In [ ]:
picked = [m for m, on in WIDGET_STATE.items() if on]
if not picked:
    picked = module_ids() if MODULES_TO_FETCH == 'ALL' else [m.upper() for m in MODULES_TO_FETCH]

keys = sources_for(picked)
print(f"Selected modules: {', '.join(picked)}")
print(f"That needs {len(keys)} file(s), {download.human(download_bytes(picked))} in total.\n")

outcomes = {}
for key in keys:
    print(f"  {SOURCES[key]['title']}")
    status, detail = download.fetch(key)
    outcomes[key] = status
    print(f"    -> {status}: {detail}\n")

if not keys:
    print('None of your chosen modules needs a download — their data is generated locally.')
failed = [key for key, status in outcomes.items() if status in ('failed', 'mismatch')]
if failed:
    print('Some downloads did not succeed:', ', '.join(failed))
    print('Try again (the servers are occasionally busy), or use the offline fallback in section 4.')
else:
    print('All downloads complete. Continue to section 4.')


## 4 · Prepare the teaching tables

The raw downloads are Excel workbooks, gzipped GEO matrices and a 7 MB association catalogue. This step turns them into the small, tidy tables the notebooks read, and generates the simulated parts for modules C and E.

**This step needs pandas, numpy and scikit-learn** — the only part of this notebook that does. If you have not installed the environment yet, that is fine: install it, come back, and re-run this one cell. (Or just skip it — the module notebooks will run the preparation themselves the first time you open them.)

### If you have no working internet at all

Ask the instructor for the `data/raw` folder on a USB stick, set `INSTRUCTOR_FOLDER` below to where you copied it, and run the cell after this one.


In [ ]:
try:
    import prepare
    for module in picked:
        try:
            print(' ', prepare.prepare(module, force=True))
        except FileNotFoundError as error:
            print(f'  {module}: skipped — {error}')
    print('\nDone. Continue to section 5.')
except ImportError as error:
    print('The teaching environment is not installed yet:', error)
    print('\nInstall it with:   conda env create -f environment.yml')
    print('             or:   pip install numpy pandas matplotlib scikit-learn jupyter')
    print('\nThen re-run this cell. Your downloads in section 3 are already safe.')


In [ ]:
# OFFLINE FALLBACK — only needed if section 3 could not download anything.
INSTRUCTOR_FOLDER = None      # e.g. Path('/media/usb/ad_practical_data/raw')

if INSTRUCTOR_FOLDER is None:
    print('No offline folder set — skip this cell unless the downloads failed for you.')
else:
    for key in sources_for(picked):
        status, detail = download.copy_from_folder(key, INSTRUCTOR_FOLDER)
        print(f"  {Path(SOURCES[key]['path']).name}: {status} — {detail}")
    print('\nNow re-run section 4 to prepare the tables.')


## 5 · Verify

One row per module. **If anything is not ✅, screenshot this table and send it to the instructor before the session** — not during it.

- ✅ everything present and verified
- ⚠️ downloaded but the prepared tables are missing (run section 4)
- ❌ raw files missing (run section 3)


In [ ]:
total_bytes = 0
print(f"{'Module':<8}{'Status':<8}Detail")
print('-' * 84)
for module in module_ids():
    entry = MODULES[module]
    raw_ok = all(download.check(key)[0] == 'ok' for key in entry['sources'])
    derived = [repo_root / name for name in entry['derived']]
    derived_ok = all(path.exists() for path in derived)
    size = sum(path.stat().st_size for path in derived if path.exists())
    size += sum((repo_root / SOURCES[key]['path']).stat().st_size
                for key in entry['sources'] if (repo_root / SOURCES[key]['path']).exists())
    total_bytes += size
    if raw_ok and derived_ok:
        mark, detail = '✅', f'ready — {download.human(size)} on disk'
    elif raw_ok:
        mark, detail = '⚠️', 'downloaded, but not prepared yet — run section 4'
    elif derived_ok:
        mark, detail = '✅', f'ready (prepared) — {download.human(size)} on disk'
    else:
        mark, detail = '❌', 'raw files missing — run section 3'
    print(f'{module:<8}{mark:<8}{detail}')
print('-' * 84)
print(f'Total on disk: {download.human(total_bytes)}')
print('\nNext: run 01_setup_check.ipynb, also at home. Then you are done until the session.')
